# Spectral clustering

In this notebook we show how to implement the different spectral clustering algorithms we have seen in the course.

In [1]:
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_mutual_info_score as ami
from scipy.sparse import diags
import sys
from sklearn.cluster import KMeans

sys.path.append('../')
from src.CD import *
from src.utils import *

import warnings
warnings.filterwarnings('ignore')

## Generate a graph with communities

First we generate a graph with communities from the stochastic block model.

In [2]:
# generate a random graph with communities

c = 8 # average degree
alpha = 0.2 # sets the level of assortativity. The smaller alpha, the stronger the community structure
c_out = alpha*c

k = 2 # number of communities
π = np.ones(k)/k # vector with the sizes of each community
C_matrix = matrix_C(c_out, c, 0, π) # generates the matrix C
n = 50000 # number of nodes
label = (k*np.arange(n)/n).astype(int) # label vector (this work only if the classes have the same size)
theta = np.ones(n) # expected degree distribution

# generate the graph
A, _ = DCSBM(C_matrix,c, label, theta)

## Construct the matrices for the different spectral algorithms

We get the $6$ matrices used to perform spectral clustering (one is simply $A$, so we do not need to compute it)

In [3]:
# graph Laplacian
d = A@np.ones(n)
D = diags(d)
L = D - A

# symmetrized normalized Laplacian
D_05 = diags(d**(-0.5))
Lsym = D_05@A@D_05

# Non-backtracking (2n x 2n version)
Bp = GetBp(A)

# Bethe-Hessian
s, _ = eigs(Bp, k = 1, which = 'LM')
r = np.sqrt(s[0].real)
Id = diags(np.ones(n))
H = (r**2-1)*Id + D - r*A

# Regularized Laplacian
Dr_05 = diags((d + r**2-1)**(-0.5))
Lr = Dr_05@A@Dr_05

## Compute the informative eigenvectors

In [4]:
_, YA = eigsh(A, k = k, which = 'LA')
_, YL = eigsh(L, k = k, which = 'SA')
_, YLsym = eigsh(Lsym, k = k, which = 'LA')
_, YB = eigs(Bp, k = k, which = 'LM')
YB = YB[:n].real
_, YH = eigsh(H, k = k, which = 'SA')
_, YLr = eigsh(Lr, k = k, which = 'LA')

## Perform clustering 

We now use *k-means* algorithm to perform clustering on the extracted eigenvectors.

In [5]:
kmeans = KMeans(n_clusters = k, random_state=0, n_init="auto")

labels_A = kmeans.fit(YA).labels_
labels_L = kmeans.fit(YL).labels_
labels_Lsym = kmeans.fit(YLsym).labels_
labels_B = kmeans.fit(YB).labels_
labels_H = kmeans.fit(YH).labels_
labels_Lr = kmeans.fit(YLr).labels_

## Evaluate the performance

We now evaluate the performance of each algorithm, by computing the adjusted mutual information of the inferred partition with the ground truth.

In [6]:
print(f'AMI for adjacency matrix: {ami(label, labels_A)}')
print(f'AMI for Laplacian matrix: {ami(label, labels_L)}')
print(f'AMI for symmetric normalized Laplacian matrix: {ami(label, labels_Lsym)}')
print(f'AMI for non-backtracking matrix: {ami(label, labels_B)}')
print(f'AMI for Bethe-Hessian matrix: {ami(label, labels_H)}')
print(f'AMI for Regularized Laplacian matrix: {ami(label, labels_Lr)}')

AMI for adjacency matrix: 0.9016731732342447
AMI for Laplacian matrix: -3.791553070739532e-15
AMI for symmetric normalized Laplacian matrix: 0.9357282410513267
AMI for non-backtracking matrix: 0.9134283126452695
AMI for Bethe-Hessian matrix: 0.922270934085364
AMI for Regularized Laplacian matrix: 0.9255046510686552
